In [2]:
import sqlite3

DB_PATH = r"C:\Users\Administrator\dep-data-engineering-jimwellbryllsantos\data\processed\final\poi_working.db"

conn = sqlite3.connect(DB_PATH)

print("Connected successfully!")

Connected successfully!


In [5]:
import pandas as pd

schema = pd.read_sql_query(
    "PRAGMA table_info(master_dataset);",
    conn
)

schema

,cid,name,type,notnull,dflt_value,pk
0,0,clean_domain,TEXT,0,None,0
1,1,land_area_sqkm,REAL,0,None,0
2,2,raw_domain_name,TEXT,0,None,0
3,3,total_population,INTEGER,0,None,0
4,4,household_population,INTEGER,0,None,0
...,...,...,...,...,...,...
98,98,aggregated_all_health_facilities_total,INTEGER,0,None,0
99,99,primary_health_per_100k,REAL,0,None,0
100,100,hospitals_per_100k,REAL,0,None,0
101,101,hospitals_per_100sqkm,REAL,0,None,0


## Business Question 1: Economic Opportunity

Which domains have the lowest poverty incidence, and how does their average annual family income compare?

In [7]:
query = """
SELECT
    clean_domain AS domain,
    family_poverty_incidence_2023_pct AS poverty_incidence_pct,
    "2023_all_income_groups" AS avg_annual_family_income
FROM master_dataset
WHERE clean_domain IS NOT NULL
  AND TRIM(clean_domain) <> ''
ORDER BY family_poverty_incidence_2023_pct ASC,
         "2023_all_income_groups" DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)

df

,domain,poverty_incidence_pct,avg_annual_family_income
0,city of san juan,0.0,842.81
1,special geographic area,0.0,0.00
2,city of muntinlupa,0.1,506.18
3,city of mandaluyong,0.3,666.02
4,city of angeles,0.3,501.93
5,ilocos norte,0.3,447.86
6,pasay city,0.5,477.31
7,city of baguio,0.5,443.20
8,city of makati,0.6,799.54
9,city of parañaque,0.6,556.45


The results identify the domains with the lowest reported family poverty incidence in 2023 and show their corresponding average annual family income. This provides a simple comparison of economic conditions across domains, while highlighting areas with relatively lower poverty levels.

Higher reported income generally coincides with stronger economic conditions, but the relationship between income and poverty is not necessarily uniform across all domains.

Note: 
* 0.0 poverty does not inherently mean absolute absence of poverty in the area
** special geographic area did not have a separate avg annual family income in the 2023 data

## Business Question 2: Educational Access

Which domains have the greatest basic-education school access per 100,000 residents?

In [9]:
query = """
SELECT
    clean_domain AS domain,
    total_population AS population,
    total_schools AS total_basic_schools,
    basic_schools_per_100k AS schools_per_100k_population,
    basic_student_teacher_ratio AS student_teacher_ratio
FROM master_dataset
WHERE clean_domain IS NOT NULL
  AND TRIM(clean_domain) <> ''
ORDER BY basic_student_teacher_ratio DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)

df

,domain,population,total_basic_schools,schools_per_100k_population,student_teacher_ratio
0,special geographic area,214703,8,3.726077,74.000000
1,maguindanao del sur,813243,382,46.972430,62.589718
2,sulu,1146097,480,41.881272,46.405814
3,maguindanao del norte,1124811,353,31.383050,45.945910
4,city of pasig,853050,187,21.921341,44.383094
5,cotabato city,0,85,0.000000,42.525926
6,city of cagayan de oro,741617,218,29.395227,42.004710
7,basilan,541947,258,47.606131,41.457523
8,cavite,4573884,1243,27.176028,40.058618
9,city of muntinlupa,552225,98,17.746390,40.043226


The results show which domains have the highest number of basic-education schools relative to their population. A higher value indicates greater school availability on a per-capita basis, which can be used as one measure of educational access. The student-teacher ratio provides additional context on the capacity of those schools to serve students.

This measure reflects school availability, not necessarily educational quality or actual accessibility, since distance, transportation, school capacity, and utilization are not captured by this query.

## Business Question 3: Healthcare Access

Which domains have the greatest hospital access per 100 km²?

In [13]:
query = """
SELECT
    clean_domain AS domain,
    total_population AS population,
    hospitals_per_100k AS hospitals_per_100k_population,
    hospitals_per_100sqkm AS hospitals_per_100_sqkm
FROM master_dataset
WHERE clean_domain IS NOT NULL
  AND TRIM(clean_domain) <> ''
ORDER BY hospitals_per_100sqkm DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)

df

,domain,population,hospitals_per_100k_population,hospitals_per_100_sqkm
0,pateros,67319,1.485465,62.143865
1,city of manila,1902590,1.366558,61.968849
2,city of pasig,853050,1.523943,41.531346
3,city of marikina,471323,1.909519,39.250669
4,city of mandaluyong,465902,0.858550,35.357283
5,city of taguig,1308085,0.917372,33.785981
6,city of las piñas,615549,1.624566,30.544178
7,pasay city,453186,1.103300,27.957154
8,city of caloocan,1712945,0.642169,20.665821
9,city of parañaque,703245,1.279782,20.078335


The results identify the domains with the greatest number of hospitals relative to population. A higher number of hospitals per 100,000 residents indicates greater hospital availability relative to the size of the population, while the spatial-density measure provides additional geographic context.

This should be interpreted as a measure of facility availability, rather than the overall quality, capacity, utilization, or travel-time accessibility of healthcare services.